In [5]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

In [6]:
import sys
import os

# Acesso aos módulos do diretório
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: C:\pod\hackathon_pod_2025


##### Carregando pacotes

In [7]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

# Funcoes customizadas
import configs.function_basic as funcoes

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.26.4


## Carregando databases

#### Book_02

In [8]:
# Carregando book_03
book_03 = pd.read_parquet(project_root/'database/processed/book_variaveis_03.parquet')
print("Book 03 data shape:", book_03.shape)

Book 03 data shape: (1280828, 96)


In [9]:
book_03.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1280828 entries, 0 to 1280827
Data columns (total 96 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   SAFRA              1280828 non-null  int64         
 1   FPD                1280828 non-null  int64         
 2   SCORE_01           1273364 non-null  float64       
 3   SCORE_02           1280256 non-null  float64       
 4   NUM_CPF            1280828 non-null  object        
 5   SCORE_RATEO        1272792 non-null  float64       
 6   SCORE_AVG          1272792 non-null  float64       
 7   SCORE_DIFF         1272792 non-null  float64       
 8   SCORE_MIN          1280828 non-null  float64       
 9   DATADENASCIMENTO   1280828 non-null  datetime64[ns]
 10  var_03             1198460 non-null  object        
 11  var_04             1280828 non-null  object        
 12  var_05             1228039 non-null  object        
 13  var_09             600120 n

#### Base Dados Telco

In [10]:
## Carregando todos arquivos em parquet de uma pasta

all_files = [os.path.join(project_root/'database/raw/bases_recarga/bases_recarga/', f) for f in os.listdir(project_root/'database/raw/bases_recarga/bases_recarga/') if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_dados_recarga = pd.concat(df_list, ignore_index=True)
print('Base Dados Recarga data shape:', df_dados_recarga.shape)

Base Dados Recarga data shape: (51684470, 24)


In [11]:
df_dados_recarga.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51684470 entries, 0 to 51684469
Data columns (total 24 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   NUM_CPF                object
 1   DW_NUM_NTC             object
 2   DAT_INSERCAO_CREDITO   object
 3   HOR_INSERCAO_CREDITO   object
 4   DW_NUM_CLIENTE         object
 5   COD_TECNOLOGIA_DW      object
 6   COD_CANAL_AQUISICAO    object
 7   COD_TIPO_CREDITO       object
 8   COD_PROMOCAO           object
 9   VAL_CREDITO_INSERIDO   object
 10  VAL_BONUS              object
 11  VAL_REAL               object
 12  COD_PLATAFORMA_ATU     object
 13  COD_STATUS_PLATAFORMA  object
 14  IND_METODO_PAGAMENTO   object
 15  DW_PLANO_TARIFACAO     object
 16  DW_TIPO_RECARGA        object
 17  DW_TIPO_INSERCAO       object
 18  DW_FORMA_PAGAMENTO     object
 19  DW_INSTITUICAO         object
 20  COD_GRUPO_CARTAO       object
 21  DSC_GRUPO_CARTAO_WPP   object
 22  FLAG_SOS               object
 23  VALOR

#### Ajuste no Dataset de Recargas

Para diminuir o tamanho da base processada e focarmos no nosso problema de negócio, iremos filtrar apenas os CPFs que estão na base do `book_03`

Também iremos criar a coluna `SAFRA` a partir da coluna `DAT_INSERCAO_CREDITO`

In [12]:
selecao_publico = book_03['NUM_CPF'].drop_duplicates()

# Selecionando na base de recarga apenas os CPFs presentes na base de score bureau movel
df_base_recarga_selecionada = df_dados_recarga[df_dados_recarga['NUM_CPF'].isin(selecao_publico)]
df_base_recarga_selecionada.head()

,NUM_CPF,DW_NUM_NTC,DAT_INSERCAO_CREDITO,HOR_INSERCAO_CREDITO,DW_NUM_CLIENTE,COD_TECNOLOGIA_DW,COD_CANAL_AQUISICAO,COD_TIPO_CREDITO,COD_PROMOCAO,VAL_CREDITO_INSERIDO,...,IND_METODO_PAGAMENTO,DW_PLANO_TARIFACAO,DW_TIPO_RECARGA,DW_TIPO_INSERCAO,DW_FORMA_PAGAMENTO,DW_INSTITUICAO,COD_GRUPO_CARTAO,DSC_GRUPO_CARTAO_WPP,FLAG_SOS,VALOR_SOS
0,89UU788W7TW,712481754,09OCT2023:00:00:00,170410,1369129520,GSM,17598,PE,-1,20.00,...,A,606234,-2,-2,-2,-2,UB,Rec.Online,0,None
1,W7T8UNX9878,739450945,12MAR2025:00:00:00,32753,1486001611,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,WT,NaoSeAplica,0,None
2,7ZTYY8W8XNY,780682139,27JAN2025:00:00:00,5813,1473175026,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,IW,NaoSeAplica,0,None
3,T97TZT9NUZU,638382020,18DEC2024:00:00:00,3354,1473618173,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,IW,NaoSeAplica,0,None
4,877W7ZX9ZU9,674107712,17DEC2024:00:00:00,21751,1469218303,GSM,-3,PE,-1,0.00,...,A,668900,-2,-2,-2,-1,WT,NaoSeAplica,0,None


In [13]:
# Criando coluna SAFRA
df_base_recarga_selecionada = funcoes.criar_coluna_safra(df_base_recarga_selecionada, 'DAT_INSERCAO_CREDITO')

c:\pod\hackathon_pod_2025\engenharia\configs\function_basic.py:251: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[coluna_datetime] = pd.to_datetime(
c:\pod\hackathon_pod_2025\engenharia\configs\function_basic.py:257: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['SAFRA'] = df[coluna_datetime].dt.strftime('%Y%m')


In [14]:
df_base_recarga_selecionada['SAFRA'].value_counts()

202411    3652353
202410    3605527
202412    3584100
202501    3171804
202408    3091771
202409    3080054
202502    2921412
202407    2885957
202503    2769663
202405    2685113
202406    2675063
202403    2638723
202404    2570062
202312    2507127
202310    2384550
202402    2367922
202401    2366130
202311    2346645
Name: SAFRA, dtype: int64

#### Ajuste dos Tipos de Dados

In [15]:
# Ajuste dos tipos de dados para agregacao
df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'] = pd.to_numeric(df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VAL_BONUS'] = pd.to_numeric(df_base_recarga_selecionada['VAL_BONUS'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VAL_REAL'] = pd.to_numeric(df_base_recarga_selecionada['VAL_REAL'], errors='coerce').fillna(0)
df_base_recarga_selecionada['FLAG_SOS'] = pd.to_numeric(df_base_recarga_selecionada['FLAG_SOS'], errors='coerce').fillna(0)
df_base_recarga_selecionada['VALOR_SOS'] = pd.to_numeric(df_base_recarga_selecionada['VALOR_SOS'], errors='coerce').fillna(0)

C:\Users\alexandre.junior\AppData\Local\Temp\ipykernel_33348\1388732397.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'] = pd.to_numeric(df_base_recarga_selecionada['VAL_CREDITO_INSERIDO'], errors='coerce').fillna(0)
C:\Users\alexandre.junior\AppData\Local\Temp\ipykernel_33348\1388732397.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_base_recarga_selecionada['VAL_BONUS'] = pd.to_numeric(df_base_recarga_selecionada['VAL_BONUS'], errors='coerce').

#### Criação das visões agregadas por SAFRA e CPF

In [16]:
df_base_recarga_selecionada_agg = df_base_recarga_selecionada.groupby(['NUM_CPF', 'SAFRA']).agg(
        QTD_REGISTROS=('NUM_CPF', 'size'),
        VAL_CREDITO_INSERIDO=('VAL_CREDITO_INSERIDO', 'sum'),
        VAL_BONUS=('VAL_BONUS', 'sum'),
        VAL_REAL=('VAL_REAL', 'sum'),
        QTD_SOS=('FLAG_SOS', 'sum'),
        VALOR_SOS=('VALOR_SOS', 'sum')
    ).reset_index()

In [17]:
df_base_recarga_selecionada_agg

,NUM_CPF,SAFRA,QTD_REGISTROS,VAL_CREDITO_INSERIDO,VAL_BONUS,VAL_REAL,QTD_SOS,VALOR_SOS
0,777777UWTYZ,202310,2,30.0,84300.0,84330.0,0,0.0
1,777777UWTYZ,202311,5,50.0,164901.0,164951.0,0,0.0
2,777777UWTYZ,202401,2,20.0,80601.0,80621.0,0,0.0
3,777777UWTYZ,202402,4,50.0,164901.0,164951.0,0,0.0
4,777777UWTYZ,202403,2,30.0,84300.0,84330.0,0,0.0
...,...,...,...,...,...,...,...,...
16167007,ZZZZZZZZY7Y,202411,47,592.9,498756.1,499349.0,1,5.0
16167008,ZZZZZZZZY7Y,202412,56,722.9,316730.9,317453.8,1,5.0
16167009,ZZZZZZZZY7Y,202501,54,637.9,338445.1,339083.0,3,25.0
16167010,ZZZZZZZZY7Y,202502,43,462.9,173662.1,174125.0,2,10.0


## Construção do Book Comportamental por SAFRA (Modelo de Crédito)

Este processo tem como objetivo a construção de um **book comportamental temporal** para modelagem de crédito, alinhado à predição de **FPD (First Payment Default)**.

### Visão Geral
Os dados comportamentais são inicialmente agregados no nível **CPF + SAFRA**, representando o comportamento observado em cada período mensal. A partir dessa base agregada, são criadas features históricas que capturam o comportamento passado do cliente em relação à **safra de referência**.

A base final do modelo é obtida por meio de um **LEFT JOIN** entre:
- **Base alvo (label)**: CPF + SAFRA com indicador FPD (0/1)
- **Base comportamental enriquecida**: histórico anterior ao mês da SAFRA

### Construção das Features Temporais
Para cada CPF, os dados são ordenados cronologicamente por SAFRA e são criadas defasagens temporais (*lags*) utilizando apenas informações do passado:

- Safra imediatamente anterior (t-1)
- Acumulado das últimas 3 safras (t-1 a t-3)
- Acumulado das últimas 6 safras (t-1 a t-6)

As defasagens são geradas via `groupby(CPF)` com `shift`, garantindo que **nenhuma informação da própria safra ou futura seja utilizada**, evitando vazamento temporal (*data leakage*).

### Robustez e Tratamento de Casos Especiais
- CPFs sem histórico anterior permanecem na base (LEFT JOIN), com valores nulos ou zerados.
- A ausência de histórico é considerada informação relevante para o modelo.
- O método é robusto a safras faltantes (meses sem registro), utilizando apenas o histórico efetivamente disponível.

### Garantias do Processo
- As features refletem exclusivamente o comportamento conhecido **até o momento da decisão de crédito**.
- A estrutura é reproduzível, auditável e adequada para uso em modelos supervisionados.
- O book final está preparado para técnicas de modelagem estatística e de machine learning.

Este desenho segue práticas consolidadas de modelagem de risco de crédito e permite expansão futura com métricas adicionais (médias, tendências, taxas e flags de histórico).


In [18]:
variaveis = [
    'QTD_REGISTROS',
    'VAL_CREDITO_INSERIDO',
    'VAL_BONUS',
    'VAL_REAL',
    'QTD_SOS',
    'VALOR_SOS'
]

df_book_04 = funcoes.criar_lags_por_safra(
    df=df_base_recarga_selecionada_agg,
    col_cpf='NUM_CPF',
    col_safra='SAFRA',
    variaveis=variaveis,
    janelas=[1, 3, 6]
)

In [19]:
df_book_04

,NUM_CPF,SAFRA,QTD_REGISTROS,VAL_CREDITO_INSERIDO,VAL_BONUS,VAL_REAL,QTD_SOS,VALOR_SOS,QTD_REGISTROS_ULT_1_SAFRAS,QTD_REGISTROS_ULT_3_SAFRAS,...,VAL_BONUS_ULT_6_SAFRAS,VAL_REAL_ULT_1_SAFRAS,VAL_REAL_ULT_3_SAFRAS,VAL_REAL_ULT_6_SAFRAS,QTD_SOS_ULT_1_SAFRAS,QTD_SOS_ULT_3_SAFRAS,QTD_SOS_ULT_6_SAFRAS,VALOR_SOS_ULT_1_SAFRAS,VALOR_SOS_ULT_3_SAFRAS,VALOR_SOS_ULT_6_SAFRAS
0,777777UWTYZ,202310,2,30.0,84300.0,84330.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,777777UWTYZ,202311,5,50.0,164901.0,164951.0,0,0.0,2.0,2.0,...,84300.0,84330.0,84330.0,84330.0,0.0,0.0,0.0,0.0,0.0,0.0
2,777777UWTYZ,202401,2,20.0,80601.0,80621.0,0,0.0,5.0,7.0,...,249201.0,164951.0,249281.0,249281.0,0.0,0.0,0.0,0.0,0.0,0.0
3,777777UWTYZ,202402,4,50.0,164901.0,164951.0,0,0.0,2.0,9.0,...,329802.0,80621.0,329902.0,329902.0,0.0,0.0,0.0,0.0,0.0,0.0
4,777777UWTYZ,202403,2,30.0,84300.0,84330.0,0,0.0,4.0,11.0,...,494703.0,164951.0,410523.0,494853.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16167007,ZZZZZZZZY7Y,202411,47,592.9,498756.1,499349.0,1,5.0,61.0,210.0,...,4437218.5,647174.0,1882072.9,4441525.8,3.0,7.0,16.0,15.0,43.0,103.0
16167008,ZZZZZZZZY7Y,202412,56,722.9,316730.9,317453.8,1,5.0,47.0,165.0,...,4285765.0,499349.0,1544227.4,4289874.3,1.0,7.0,11.0,5.0,43.0,63.0
16167009,ZZZZZZZZY7Y,202501,54,637.9,338445.1,339083.0,3,25.0,56.0,164.0,...,3689466.9,317453.8,1463976.8,3693629.2,1.0,5.0,10.0,5.0,25.0,58.0
16167010,ZZZZZZZZY7Y,202502,43,462.9,173662.1,174125.0,2,10.0,54.0,157.0,...,3033943.4,339083.0,1155885.8,3037958.7,3.0,5.0,12.0,25.0,35.0,78.0


In [20]:
book_04 = df_book_04.copy()

In [22]:
df_book_04 = pd.merge(
    book_03,
    df_book_04,
    on=['NUM_CPF', 'SAFRA'],
    how='left'
)

In [23]:
# Sanity check
book_03.shape[0] == df_book_04.shape[0]

True

In [24]:
book_04.to_parquet(project_root/'database/processed/book_variaveis_04.parquet', index=False)